# Assignment 1 — Text Preprocessing and Bag-of-Words Classification

**344.175 Natural Language Processing — Exercise Track**
Total points: **50** · Deadline: *30.10.2026 10:00*

---

## What you will build

The classical NLP pipeline we know from scratch:

```
raw text → preprocessing → dictionary → document-word matrix → classifier → evaluation
```

The data is the **Hateful Memes** dataset: memes, each a picture with a line of text, labeled
hateful or not. Assignments 1 and 2 use the text alone; Assignments 3 and 4 add
the pictures.

## Content warning

The dataset contains real hateful content: racism, sexism, religious hatred, slurs and attacks
on people with disabilities, and some of the examples this notebook prints are offensive. Use
the data only for this course and do not redistribute it.

## Where and how to work

You work in teams of two and hand in **one** notebook per team, named with both matriculation
numbers: `assignment_01_<matriculation 1>_<matriculation 2>.ipynb`, for example `assignment_01_01234567_12345678.ipynb` (leading zeros
included). One of you uploads it in Moodle, and both of you get the grade.

- **Google Colab:** *File → Upload notebook* (the notebook from the ZIP), then *File → Rename*.
  The setup cell below installs what Colab is missing, and the data is downloaded for you.
- **Your own computer:** follow `SETUP.md` (`uv sync`, then `uv run jupyter lab`), then
  *File → Save Notebook As…*.

Go through the notebook from top to bottom. Replace every `### YOUR CODE HERE` /
`raise NotImplementedError()` with your own code and run the test cell below it:
`Task N.N passed` means the public tests pass. Before submitting, restart the kernel, run all
cells, and run the submission check at the end.

## Rules

The notebook is graded by a program that runs it and calls your functions by name.

- Do not rename functions or change their parameters.
- Do not add to, remove or modify the given cells; the grader restores them before grading.
  You may add your own cells for experiments.
- Do not import extra packages; the grader has only what `pyproject.toml` installs.
- The notebook must run from a fresh kernel without errors. A crash costs every point after it.

## Grading

```
your points = coding points (0-50) × average of the three written weights (0.0-1.0)
```

**Coding tasks: 50 points, graded automatically.** 16 tasks, each all-or-nothing. After the
deadline your functions are run on inputs you have not seen.

| Part | Topic | Points |
|---|---|---|
| 1 | Getting the data | 2 |
| 2 | Understanding the data | 10 |
| 3 | Preprocessing | 11 |
| 4 | Dictionary and Bag of Words | 15 |
| 5 | Classification and evaluation | 12 |
| | **Total** | **50** |

**Written blocks: a multiplier between 0.0 and 1.0.** Task 2.4, Task 3.5 and Part 6 are read by
a human and each scored between 0.0 and 1.0. Their average multiplies your coding points, so
50 coding points with empty written answers is 0. The questions ask for your own numbers,
examples and decisions: code can be copied, a judgment about your own output cannot. An answer
that would be equally true of any dataset earns 0.0.

## Weight in the course grade

The course has 1000 points: 50 for each of the four assignments (200 in total, at home, any
tools allowed) and 800 for the exercise exam (supervised). The exam must be passed on its own. Using AI tools on the assignments is
allowed, but the exam covers the same skills without them, so make sure you understand what
you hand in.


---

## Part 0 — Check your setup (0 points, but do it first)

Run the next cell. On Colab it installs the missing packages; on your own computer it only
checks that everything is present. Continue once it prints `Setup OK`.


In [ ]:
# --- Environment setup (works both on Colab and on your own computer) ---
import importlib
import subprocess
import sys

try:
    import google.colab          # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED = ["numpy", "pandas", "scipy", "matplotlib", "sklearn", "nltk", "certifi"]
PIP_NAME = {"sklearn": "scikit-learn"}

missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing and IN_COLAB:
    packages = [PIP_NAME.get(m, m) for m in missing]
    print(f"Colab detected — installing {', '.join(packages)} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
elif missing:
    raise SystemExit(
        f"Missing packages: {', '.join(missing)}.\n"
        "You are not running in the course environment. Start Jupyter with:\n"
        "    uv run jupyter lab\n"
        "See SETUP.md."
    )

print("Running on:", "Google Colab" if IN_COLAB else "your own computer")

In [ ]:
import json
import re
import ssl
import urllib.error
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"Python {sys.version.split()[0]}")
print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")

# Random seed. Do not change this number.
SEED = 42
np.random.seed(SEED)

# Show all columns of wide tables and do not cut long meme texts.
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 120)

print("\nSetup OK")

### NLTK resources

`nltk` (Natural Language Toolkit) provides the stop word list and the WordNet dictionary used
in Part 3. Both have to be downloaded once; the next cell does that. It needs internet the
first time and prints `already there` on later runs.


In [ ]:
import ssl

import nltk

# Some Python installations cannot verify HTTPS certificates, which makes the NLTK
# download fail. certifi provides the missing certificate list.
try:
    import certifi
    ssl._create_default_https_context = lambda *a, **kw: ssl.create_default_context(
        cafile=certifi.where()
    )
except ImportError:
    pass

def _nltk_present(path):
    """NLTK finds an extracted corpus at `path` and a still-zipped one at `path.zip`."""
    for candidate in (path, path + ".zip"):
        try:
            nltk.data.find(candidate)
            return True
        except LookupError:
            pass
    return False


for resource, path in [("stopwords", "corpora/stopwords"),
                       ("wordnet", "corpora/wordnet"),
                       ("omw-1.4", "corpora/omw-1.4")]:
    if _nltk_present(path):
        print(f"  {resource:<12} already there")
        continue
    print(f"  {resource:<12} downloading ...")
    if not nltk.download(resource, quiet=True) or not _nltk_present(path):
        raise RuntimeError(
            f"Could not download the NLTK resource {resource!r}. Check your internet "
            "connection and run this cell again. On a computer without internet, copy an "
            "`nltk_data` folder over from one that has it (see SETUP.md)."
        )

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Quick check that both resources actually load.
assert WordNetLemmatizer().lemmatize("wolves") == "wolf", "WordNet did not load"
print("\nNLTK OK —", len(stopwords.words("english")), "English stop words available")

---

# Part 1 — Getting the data (2 points)

## The Hateful Memes dataset

Published by Meta AI in 2020 (Kiela et al., *The Hateful Memes Challenge: Detecting Hate Speech
in Multimodal Memes*, NeurIPS 2020). One design decision matters for everything that follows:
the dataset contains **benign confounders**. For a hateful meme the authors also built a
harmless one by swapping the image or the text. So *"look how many people love you"* is hateful
on a picture of an empty desert and harmless on a picture of a crowd.

## The files

Five splits in **JSON Lines** format (`.jsonl`, one JSON object per line):

| file | rows | what it is |
|---|---|---|
| `train.jsonl` | 8500 | training set, everything is learned from here |
| `dev_seen.jsonl` | 500 | validation set, for choosing hyperparameters |
| `dev_unseen.jsonl` | 540 | a second, harder validation set |
| `test_seen.jsonl` | 1000 | test set, touch only at the very end |
| `test_unseen.jsonl` | 2000 | a second, harder test set |

The next cell downloads them (about 1.5 MB) into a `data` folder next to your notebook. On
Colab, run it again whenever the runtime has been recycled.


In [ ]:
# --- Downloading the data (nothing to change here) ---
BASE_URL = "https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main"

# split name -> (file name, how many lines it must have)
SPLITS = {
    "train":       ("train.jsonl",       8500),
    "dev_seen":    ("dev_seen.jsonl",     500),
    "dev_unseen":  ("dev_unseen.jsonl",   540),
    "test_seen":   ("test_seen.jsonl",   1000),
    "test_unseen": ("test_unseen.jsonl", 2000),
}
DATA_DIR = Path("data")


def _ssl_context():
    """Some Python installations cannot verify HTTPS certificates; certifi fixes that."""
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        return ssl.create_default_context()


def download_hateful_memes(data_dir=DATA_DIR):
    """Make sure all five .jsonl files are present. Safe to run again: it skips what is there."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    for filename, n_expected in SPLITS.values():
        target = data_dir / filename
        if target.exists() and sum(1 for _ in target.open(encoding="utf-8")) == n_expected:
            print(f"  {filename:<20} already there ({n_expected} lines)")
            continue

        print(f"  {filename:<20} downloading ...", end=" ", flush=True)
        try:
            request = urllib.request.Request(f"{BASE_URL}/{filename}",
                                             headers={"User-Agent": "nlp-uebung/1.0"})
            with urllib.request.urlopen(request, context=_ssl_context(), timeout=60) as response:
                target.write_bytes(response.read())
        except (urllib.error.URLError, OSError) as err:
            target.unlink(missing_ok=True)
            raise RuntimeError(
                f"Download failed: {err}\n\n"
                f"Download the five files by hand from {BASE_URL}/<filename> "
                f"and put them into {data_dir.resolve()}, then run this cell again."
            ) from err

        n_got = sum(1 for _ in target.open(encoding="utf-8"))
        if n_got != n_expected:
            target.unlink(missing_ok=True)
            raise RuntimeError(f"{filename} has {n_got} lines instead of {n_expected} — "
                               "the download broke off. Run the cell again.")
        print(f"ok ({n_got} lines)")
    return data_dir


def load_split(split, data_dir=DATA_DIR):
    """Read one split into a list of dictionaries."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split {split!r}. Choose one of: {list(SPLITS)}")
    path = Path(data_dir) / SPLITS[split][0]
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


data_dir = download_hateful_memes()
print(f"\nData is in: {data_dir.resolve()}")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 0 POINTS &middot; NOT GRADED</span>

## Task 1.0 — Look at the raw data (0 points)

Before loading anything into a table, look at the file itself.

In the cell below:
1. open `data/train.jsonl`, read its first line and print it exactly as it is stored;
2. print `load_split("train")[0]`, the same line parsed into a Python dictionary.

Then read what you printed: which fields does one example have, and what does each hold? Nothing is graded here, but you need this picture of the data for Task 1.1.

</div>


In [ ]:
# Print the first line of data/train.jsonl as it is stored, then the same line as a dictionary.
### YOUR CODE HERE



<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 2 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 1.1 — Load the splits into DataFrames (2 points)

The fields you just printed:

| field | meaning |
|---|---|
| `id` | unique number of the meme |
| `img` | path of the picture in the original dataset (not needed until Assignment 3) |
| `label` | **0 = not hateful**, **1 = hateful** |
| `text` | the text written on the meme, our input |

A list of dictionaries is awkward to analyze, so convert it into a pandas DataFrame: a table
with named columns.

Write `load_dataframe(split)` so that it
1. calls `load_split(split)` to get the list of dictionaries,
2. turns it into a `pd.DataFrame`,
3. keeps only the columns `id`, `text`, `label`, in exactly that order,
4. returns that DataFrame.

</div>


In [ ]:
def load_dataframe(split):
    """Load one split of the Hateful Memes data as a DataFrame with columns id, text, label."""
    ### YOUR CODE HERE
    raise NotImplementedError()


df_train = load_dataframe("train")
df_dev = load_dataframe("dev_seen")
df_test = load_dataframe("test_seen")

df_train.head(4)

In [ ]:
# --- Test 1.1: do not add, remove or modify anything in this cell ---
assert isinstance(df_train, pd.DataFrame), "load_dataframe must return a pandas DataFrame."
assert list(df_train.columns) == ["id", "text", "label"], (
    f"Columns must be exactly ['id', 'text', 'label'], got {list(df_train.columns)}."
)
assert len(df_train) == 8500, f"train should have 8500 rows, got {len(df_train)}."
assert len(df_dev) == 500, f"dev_seen should have 500 rows, got {len(df_dev)}."
assert len(df_test) == 1000, f"test_seen should have 1000 rows, got {len(df_test)}."
assert df_train["label"].isin([0, 1]).all(), "Labels must all be 0 or 1."
assert isinstance(df_train["text"].iloc[0], str), "The text column should contain strings."
print("Task 1.1 passed")

### Missing values and leakage (nothing to implement)

Two problems silently ruin results: missing values, and **leakage**, the same meme in training
and test, which the model can then memorize. The next cell checks both for the three splits we
use. The two `unseen` files are left out: `dev_unseen` shares 400 memes with `dev_seen` on
purpose.


In [ ]:
# --- Missing values and leakage (nothing to change here) ---
SPLITS_USED = {"train": df_train, "dev_seen": df_dev, "test_seen": df_test}

quality = pd.DataFrame({
    name: {
        "rows": len(df),
        "missing_text": int((df["text"].fillna("").str.strip() == "").sum()),
        "missing_label": int(df["label"].isna().sum()),
        "duplicate_ids": int(df["id"].duplicated().sum()),
    }
    for name, df in SPLITS_USED.items()
}).T
display(quality)

for a, b in [("train", "dev_seen"), ("train", "test_seen"), ("dev_seen", "test_seen")]:
    shared = set(SPLITS_USED[a]["id"]) & set(SPLITS_USED[b]["id"])
    print(f"memes in both {a} and {b}: {len(shared)}")


---

# Part 2 — Understanding the data (10 points + written block 1)

Before modeling anything, look at the data. Every number you compute here helps you interpret
your classifier later.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 2.1 — Label distribution and the majority baseline (3 points)

How many examples per class? And how well does a model do that learns nothing and always
predicts the most frequent class? That is the **majority baseline**, the number your real
model has to beat. 64% accuracy sounds fine until you learn that always saying "not hateful"
also gets 64%.

Write two functions:
- `label_distribution(df)` → `{0: count_of_class_0, 1: count_of_class_1}` with plain Python `int`s. Both keys must always be present; a class that does not occur gets the count `0`.
- `majority_baseline_accuracy(df_train, df_eval)` → the accuracy on `df_eval` when you always predict the class that is most frequent in `df_train`. If both classes are exactly equally frequent in `df_train`, predict class `0`.

Note the asymmetry in the second function: the majority class is learned from the training set
and evaluated on the evaluation set. Using the evaluation labels to decide what to predict
would be cheating. The tie rule never triggers on the real training set, but the hidden tests
include a tied toy example.

</div>


In [ ]:
def label_distribution(df):
    """Return {0: n_not_hateful, 1: n_hateful} for the given DataFrame."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def majority_baseline_accuracy(df_train, df_eval):
    """Accuracy on df_eval when always predicting the majority class of df_train."""
    ### YOUR CODE HERE
    raise NotImplementedError()


for name, df in [("train", df_train), ("dev_seen", df_dev), ("test_seen", df_test)]:
    dist = label_distribution(df)
    total = dist[0] + dist[1]
    print(f"{name:<10} not hateful: {dist[0]:>5} ({dist[0]/total:6.1%})   "
          f"hateful: {dist[1]:>5} ({dist[1]/total:6.1%})")

print()
print(f"Majority baseline on dev : {majority_baseline_accuracy(df_train, df_dev):.1%}")
print(f"Majority baseline on test: {majority_baseline_accuracy(df_train, df_test):.1%}")

In [ ]:
# --- Test 2.1: do not add, remove or modify anything in this cell ---
dist = label_distribution(df_train)
assert dist == {0: 5481, 1: 3019}, f"Expected {{0: 5481, 1: 3019}}, got {dist}."
assert isinstance(dist[0], int), "The counts should be plain Python ints."

assert label_distribution(pd.DataFrame({"label": [0, 0]})) == {0: 2, 1: 0}, (
    "Both keys must be present even when a class never occurs: expected {0: 2, 1: 0}."
)
toy_train = pd.DataFrame({"label": [0, 0, 0, 1]})     # majority class is 0
toy_eval = pd.DataFrame({"label": [0, 1, 1, 1]})      # predicting 0 is right once in four
got = majority_baseline_accuracy(toy_train, toy_eval)
assert abs(got - 0.25) < 1e-9, f"Expected 0.25 on the toy example, got {got}."

dev_base = majority_baseline_accuracy(df_train, df_dev)
assert abs(dev_base - 0.506) < 1e-6, f"Expected 0.506 on dev, got {dev_base}."
print("Task 2.1 passed")

The training set is imbalanced (about 64% / 36%), but dev and test are almost perfectly
balanced. The dataset authors did that on purpose. So the majority baseline on dev and test is
about 50%, and that is the number to compare every accuracy against, not 64%.


### How long are the texts? (nothing to implement)

Document length decides how much length normalization (in `tf` and BM25) matters and how
sparse the document vectors will be. "Words" here are just `text.split()`; proper tokenization
comes in Part 3.


In [ ]:
# --- Text length (nothing to change here) ---
def length_stats(df):
    n_words = df["text"].str.split().str.len()
    return {"mean_chars": df["text"].str.len().mean(), "mean_words": n_words.mean(),
            "median_words": n_words.median(), "min_words": n_words.min(),
            "max_words": n_words.max()}


display(pd.DataFrame({name: length_stats(df) for name, df in SPLITS_USED.items()}).T.round(2))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df_train["text"].str.split().str.len(), bins=40, color="#4C72B0", edgecolor="white")
ax.set_xlabel("number of words per meme text")
ax.set_ylabel("number of memes")
ax.set_title("Text length in the training set")
plt.tight_layout()
plt.show()


The texts are short: about 12 words on average, median 10, longest 70. A single informative
word can therefore dominate a document vector, and length normalization matters less here than
it would for news articles.

### The most frequent words (nothing to implement)

The next cell counts how often each lowercased word occurs in total (not in how many documents;
that is document frequency and comes in Part 4).


In [ ]:
# --- Word counts (nothing to change here) ---
def word_counts(texts):
    """Count how often each lowercased whitespace token occurs across all texts."""
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())
    return counter


train_counts = word_counts(df_train["text"])
print(f"{len(train_counts)} different words in the training set\n")
print(f"{'word':<12}{'count':>8}")
for word, count in train_counts.most_common(20):
    print(f"{word:<12}{count:>8}")


The top-20 list is *the, a, you, to, i, and, when, is, of, …*: the **stop words** from
Lecture 2. They dominate the counts and say nothing about hatefulness, which is the motivation
for stop word removal and idf weighting.

### Background: Zipf's law (nothing to implement)

Not part of the lecture, but it explains why Parts 3 and 4 exist: word frequencies follow a
power law, which is a straight line when frequency is plotted against rank on log-log axes.


In [ ]:
ranked = train_counts.most_common()
frequencies = np.array([count for _, count in ranked])
ranks = np.arange(1, len(frequencies) + 1)

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.loglog(ranks, frequencies, color="#4C72B0", linewidth=2)
for rank in [1, 10, 100, 1000]:
    word, count = ranked[rank - 1]
    left.plot(rank, count, "o", color="#C44E52", markersize=7)
    left.annotate(f"  {word} ({count})", (rank, count), fontsize=9, color="#C44E52")
left.set_xlabel("rank of the word (log)")
left.set_ylabel("how often it occurs (log)")
left.set_title("Zipf's law: frequency against rank")
left.grid(alpha=0.3, which="both")

# How much of the corpus do the k most frequent words account for?
cumulative = np.cumsum(frequencies) / frequencies.sum()
right.plot(ranks, cumulative * 100, color="#55A868", linewidth=2)
for k in [10, 100, 1000]:
    right.plot(k, cumulative[k - 1] * 100, "o", color="#C44E52", markersize=7)
    right.annotate(f"  top {k} words = {cumulative[k-1]:.0%} of all text",
                   (k, cumulative[k - 1] * 100), fontsize=9, color="#C44E52")
right.set_xscale("log")
right.set_xlabel("number of most frequent words kept (log)")
right.set_ylabel("% of all tokens covered")
right.set_title("Coverage of the k most frequent words")
right.set_ylim(0, 100)
right.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print(f"vocabulary: {len(frequencies):,} distinct words, {frequencies.sum():,} tokens")
print(f"words occurring exactly once: {(frequencies == 1).sum():,} "
      f"({(frequencies == 1).mean():.0%} of the vocabulary)")

Left: the most frequent word occurs thousands of times, and about half of the vocabulary
occurs exactly once. Right: a few hundred words cover most of the running text, and the most
frequent of them, *the, a, you, to*, say nothing about hatefulness. A dictionary built from raw counts therefore spends most of
its dimensions on words seen once, while the constant words dominate every document vector.
`min_count`, stop word removal and idf are the three tools from Lecture 2 against this.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 2.2 — Which words point at which class? (4 points)

Which words appear disproportionately in hateful texts, and which in non-hateful ones? A
simple and robust measure is the **smoothed ratio of relative frequencies**. For a word $v$:

$$\text{score}(v) = \frac{\big(\text{count of } v \text{ in class 1} + \alpha\big) \big/ \big(\text{total words in class 1} + \alpha N\big)}{\big(\text{count of } v \text{ in class 0} + \alpha\big) \big/ \big(\text{total words in class 0} + \alpha N\big)}$$

- A score above 1 means the word is relatively more common in hateful texts, below 1 in
  non-hateful texts.
- $\alpha$ (here $1$) is Laplace smoothing (Lecture 4). Without it, a word with 0 occurrences
  in class 0 would divide by zero.
- $N$ is the number of distinct words across both classes.

Write `class_association(df, alpha=1.0, min_total_count=10)` returning a DataFrame with the word as index and the columns `count_0` (occurrences in non-hateful texts), `count_1` (occurrences in hateful texts) and `score` (the formula above), sorted by `score` descending. Include only words that occur at least `min_total_count` times in total.

The top of this list is genuinely offensive. That is the nature of the task.

</div>


In [ ]:
def class_association(df, alpha=1.0, min_total_count=10):
    """Words ranked by how much more often they occur in hateful vs non-hateful texts."""
    ### YOUR CODE HERE
    raise NotImplementedError()


assoc = class_association(df_train)
print("Most strongly associated with HATEFUL (label 1):")
display(assoc.head(15))
print("Most strongly associated with NOT HATEFUL (label 0):")
display(assoc.tail(15).iloc[::-1])

In [ ]:
# --- Test 2.2: do not add, remove or modify anything in this cell ---
assert list(assoc.columns) == ["count_0", "count_1", "score"], (
    f"Columns must be ['count_0', 'count_1', 'score'], got {list(assoc.columns)}."
)
assert assoc["score"].is_monotonic_decreasing, "The DataFrame must be sorted by score, descending."
assert ((assoc["count_0"] + assoc["count_1"]) >= 10).all(), "Words with fewer than 10 occurrences must be dropped."

# A toy corpus where we know the answer: 'x' only ever appears in class 1.
toy = pd.DataFrame({
    "text": ["x x x x x y y y y y", "x x x x x y y y y y", "y y y y y y y y y y", "y y y y y y y y y y"],
    "label": [1, 1, 0, 0],
})
ta = class_association(toy)
assert ta.index[0] == "x", f"'x' should rank first for class 1, got '{ta.index[0]}'."
assert ta.loc["x", "count_1"] == 10 and ta.loc["x", "count_0"] == 0
assert ta.loc["x", "score"] > 1 > ta.loc["y", "score"], "Scores are on the wrong side of 1."
print("Task 2.2 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 2.3 — Same text, opposite label (3 points)

For a hateful meme, the authors often built a benign confounder with the same text on a
different picture. If the same sentence appears once labeled hateful and once labeled not
hateful, a model that sees only the text gives both the same answer, so it gets at most one
of them right. The picture tells them apart; the text cannot.

Count how much of the training set is affected:

1. Group the documents by their text (lowercased and stripped, so that `"Hello "` and
   `"hello"` count as the same text).
2. A group is **conflicting** if it contains both labels.

Write `conflicting_texts(df)` returning a dictionary with the keys
- `"n_distinct_texts"`: how many distinct texts there are
- `"n_conflicting_texts"`: how many distinct texts are conflicting
- `"n_conflicting_docs"`: how many documents sit in a conflicting group

</div>


In [ ]:
def conflicting_texts(df):
    """Count the texts that occur with both labels."""
    ### YOUR CODE HERE
    raise NotImplementedError()

conflicts = conflicting_texts(df_train)
print(f"distinct texts             : {conflicts['n_distinct_texts']:,} "
      f"(out of {len(df_train):,} documents)")
print(f"texts with both labels     : {conflicts['n_conflicting_texts']:,}")
print(f"documents with a conflict  : {conflicts['n_conflicting_docs']:,} "
      f"({conflicts['n_conflicting_docs'] / len(df_train):.1%} of the training set)")

In [ ]:
# --- Test 2.3: do not add, remove or modify anything in this cell ---
# "same text" and "SAME TEXT " are one text with both labels: 1 conflicting text, 2 documents.
toy = pd.DataFrame({"text": ["same text", "SAME TEXT ", "different"], "label": [0, 1, 1]})
got = conflicting_texts(toy)
assert set(got) == {"n_distinct_texts", "n_conflicting_texts", "n_conflicting_docs"}, (
    f"The dictionary must have exactly the three required keys, got {sorted(got)}."
)
assert got["n_distinct_texts"] == 2, f"'same text' and 'SAME TEXT ' are one group -> 2, got {got['n_distinct_texts']}"
assert got["n_conflicting_texts"] == 1, f"one text has both labels -> 1, got {got['n_conflicting_texts']}"
assert got["n_conflicting_docs"] == 2, f"both documents of the conflicting group count -> 2, got {got['n_conflicting_docs']}"

same = conflicting_texts(pd.DataFrame({"text": ["a", "a", "b"], "label": [1, 1, 0]}))
assert same["n_conflicting_texts"] == 0 and same["n_conflicting_docs"] == 0, (
    "a repeated text with the same label is not a conflict"
)

real = conflicting_texts(df_train)
assert real["n_distinct_texts"] == 7072, f"expected 7072 distinct texts, got {real['n_distinct_texts']}"
assert real["n_conflicting_texts"] == 636, f"expected 636, got {real['n_conflicting_texts']}"
assert real["n_conflicting_docs"] == 1524, f"expected 1524, got {real['n_conflicting_docs']}"
print("Task 2.3 passed")

Almost one in five training documents has a twin with the same text and the opposite label.
A text-only model must give both twins the same answer, so it gets at most one of each pair
right, and during training these documents contradict each other. This counts only exact
duplicates. Near-duplicates and texts that are ambiguous without the picture add to the
problem.

The next cell prints some of the conflicting pairs.


In [ ]:
groups = {}
for text, label, meme_id in zip(df_train["text"], df_train["label"], df_train["id"]):
    groups.setdefault(text.strip().lower(), []).append((label, meme_id))

conflicting = [(text, entries) for text, entries in groups.items()
               if len({label for label, _ in entries}) > 1]

print(f"{len(conflicting)} texts appear with both labels. Three of them:\n")
for text, entries in conflicting[:3]:
    print(f'  "{text[:88]}"')
    for label, meme_id in sorted(entries)[:4]:
        print(f"      id {meme_id}: label {label}  ({'hateful' if label else 'not hateful'})")
    print()

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">WRITTEN ANSWER &middot; BLOCK 1 OF 3 &middot; GRADED BY A HUMAN</span>

Scored between **0.0** and **1.0** (see *Grading* at the top). Quote *your* numbers and examples from this notebook; statements that would be true of any dataset score 0.0.

## Task 2.4 — What have you seen so far?

Two questions. Answer each in 3 to 6 sentences; question 2 starts with a single number.

### Question 1 — One conflicting pair

The cell above printed texts that occur in the training set with both labels. Pick one and
copy the text and both meme ids into your answer, exactly as printed. Then: what would the two
images have to show for both labels to be correct? And, reading only the text, which label
would you have given, and how sure would you have been?

### Question 2 — Your prediction, before you build anything

Write down one number: the test accuracy you expect from the classifier in Part 5. Add one
sentence on why that number rather than ten points higher. Part 6 asks you to come back to it,
so do not change it later. You are graded on the reasoning, not on being right.

</div>


<div style="background:#EEF5FB;border:1px solid #4C72B0;border-left:9px solid #4C72B0;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#2A5183;letter-spacing:.05em;font-size:.9em">YOUR ANSWERS</span> &mdash; replace every *(write here)* line and keep the headings.

### Task 2.4 — question 1  (the conflicting pair)

**Text:** _(write here)_

**Meme ids:** _(write here)_

_(write here: what the two images would have to show, and which label you would have given)_

### Task 2.4 — question 2  (your prediction)

**My predicted test accuracy:** _(write one number here, e.g. 61%)_

_(write here: one sentence on why that number)_

</div>


---

# Part 3 — Preprocessing (11 points + written block 2)

The preprocessing chain from Lecture 2:

```
raw text → normalize → tokenize → remove stop words → stem or lemmatize → tokens
```

Every step has the same goal: different surface forms of the same thing should collapse into
one dictionary entry. That shrinks the vocabulary and reduces sparsity.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 3.1 — Normalization (4 points)

Write `normalize_text(text)` that applies, in this order:
1. **case folding**: lowercase everything;
2. **URLs** (http, https or www addresses) → the token `<url>`;
3. **numbers** (a run of digits, possibly with `.` or `,` inside, like `2020`, `3.5`, `1,000`) → the token `<num>`;
4. **punctuation removal**: delete every character that is not a letter, a digit, whitespace, or `<` / `>` (the brackets of the special tokens must survive);
5. **whitespace collapsing**: any run of whitespace becomes one space; strip both ends.

In steps 2 and 3 insert the token with a space on each side, `" <url> "` and `" <num> "`. Otherwise `12/34-56` loses its separators in step 4 and fuses into `<num><num><num>`. With the padding it becomes three tokens, and step 5 removes the extra spaces again.

Examples of what must come out:

| input | output |
|---|---|
| `"Hello,   WORLD!"` | `"hello world"` |
| `"In 2020 I had 3.5 apples."` | `"in <num> i had <num> apples"` |
| `"see http://example.com/x now"` | `"see <url> now"` |
| `"U.S.A. rules"` | `"usa rules"` |

</div>


In [ ]:
URL_PATTERN = r"https?://\S+|www\.\S+"
NUMBER_PATTERN = r"\d+(?:[.,]\d+)*"


def normalize_text(text):
    """Lowercase, replace URLs and numbers, drop punctuation, collapse whitespace."""
    ### YOUR CODE HERE
    raise NotImplementedError()


for example in ["Hello,   WORLD!", "In 2020 I had 3.5 apples.",
                "see http://example.com/x now", "U.S.A. rules"]:
    print(f"{example!r:<40} -> {normalize_text(example)!r}")

In [ ]:
# --- Test 3.1: do not add, remove or modify anything in this cell ---
cases = {
    "Hello,   WORLD!": "hello world",
    "In 2020 I had 3.5 apples.": "in <num> i had <num> apples",
    "see http://example.com/x now": "see <url> now",
    "U.S.A. rules": "usa rules",
    "  spaces   everywhere  ": "spaces everywhere",
    "1,000 dollars": "<num> dollars",
    "Room 12/34-56 please": "room <num> <num> <num> please",
    "": "",
}
for raw, expected in cases.items():
    got = normalize_text(raw)
    assert got == expected, f"normalize_text({raw!r})\n  expected {expected!r}\n  got      {got!r}"
print("Task 3.1 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 2 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 3.2 — Tokenization and stop words (2 points)

After normalization the text is clean enough that splitting on whitespace is a correct
tokenizer.

Write two small functions:
- `tokenize(text)`: normalize the text, then split it into a list of tokens.
- `remove_stopwords(tokens)`: drop every token in the NLTK English stop word list (`STOPWORDS`). Return a new list and leave the input untouched. `<num>` and `<url>` are not in that list, so they survive.

</div>


In [ ]:
STOPWORDS = set(stopwords.words("english"))


def tokenize(text):
    """Normalize the text and split it into a list of tokens."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def remove_stopwords(tokens):
    """Return the tokens that are not English stop words."""
    ### YOUR CODE HERE
    raise NotImplementedError()


example = df_train["text"].iloc[3]
print("raw:              ", example)
print("tokenized:        ", tokenize(example))
print("without stopwords:", remove_stopwords(tokenize(example)))

In [ ]:
# --- Test 3.2: do not add, remove or modify anything in this cell ---
assert tokenize("Hello, WORLD!") == ["hello", "world"]
assert tokenize("") == [], "An empty string must give an empty list of tokens."
assert tokenize("In 2020!") == ["in", "<num>"]
assert remove_stopwords(["the", "cat", "is", "here"]) == ["cat"], (
    f"got {remove_stopwords(['the', 'cat', 'is', 'here'])}"
)
assert remove_stopwords(["<num>", "<url>"]) == ["<num>", "<url>"], "Special tokens must survive."
assert remove_stopwords([]) == []
_original = ["the", "dog"]
remove_stopwords(_original)
assert _original == ["the", "dog"], "remove_stopwords must return a NEW list and leave its input untouched."
print("Task 3.2 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 3.3 — Stemming and lemmatization (3 points)

Both reduce inflected forms to a common form (Lecture 2). A **stemmer** chops affixes off with
rules: fast, no dictionary, and the result is often not a real word (`automation → autom`). A
**lemmatizer** looks the word up in WordNet and returns the real base form
(`children → child`), but fails on words it does not know.

Write `stem_tokens(tokens)` and `lemmatize_tokens(tokens)` using the `stemmer` and `lemmatizer` objects created in the next cell. Leave the lemmatizer's part of speech at its default: it treats every word as a noun, so `cars` becomes `car` while `running` stays `running`.

</div>


In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def stem_tokens(tokens):
    """Apply the Porter stemmer to every token."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def lemmatize_tokens(tokens):
    """Apply the WordNet lemmatizer to every token."""
    ### YOUR CODE HERE
    raise NotImplementedError()


demo = ["automation", "automates", "cars", "children", "is", "better", "studies"]
print(f"{'token':<12}{'stemmed':<12}{'lemmatized':<12}")
for token in demo:
    print(f"{token:<12}{stem_tokens([token])[0]:<12}{lemmatize_tokens([token])[0]:<12}")

In [ ]:
# --- Test 3.3: do not add, remove or modify anything in this cell ---
assert stem_tokens(["automation", "automates"]) == ["autom", "autom"], (
    f"got {stem_tokens(['automation', 'automates'])}"
)
assert stem_tokens(["compressed", "compression"]) == ["compress", "compress"]
assert lemmatize_tokens(["cars", "children"]) == ["car", "child"], (
    f"got {lemmatize_tokens(['cars', 'children'])}"
)
assert stem_tokens([]) == [] and lemmatize_tokens([]) == []
print("Task 3.3 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 2 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 3.4 — Compare the pipelines (2 points)

Measure what each step buys you. Four pipelines, compared by vocabulary size on the training
set:

| name | steps |
|---|---|
| `raw` | tokenize only |
| `no_stop` | tokenize → remove stop words |
| `stemmed` | tokenize → remove stop words → stem |
| `lemmatized` | tokenize → remove stop words → lemmatize |

Write `preprocess(text, remove_stops=False, mode="none")`, where `mode` is one of `"none"`, `"stem"`, `"lemma"`, returning the final list of tokens. The rest of the cell fills the comparison table. Stemming 8500 texts takes a few seconds.

</div>


In [ ]:
def preprocess(text, remove_stops=False, mode="none"):
    """Full preprocessing chain. mode is one of "none", "stem", "lemma"."""
    ### YOUR CODE HERE
    raise NotImplementedError()


PIPELINES = {
    "raw":        dict(remove_stops=False, mode="none"),
    "no_stop":    dict(remove_stops=True,  mode="none"),
    "stemmed":    dict(remove_stops=True,  mode="stem"),
    "lemmatized": dict(remove_stops=True,  mode="lemma"),
}

rows = []
train_tokenized = {}
for name, kwargs in PIPELINES.items():
    docs = [preprocess(t, **kwargs) for t in df_train["text"]]
    train_tokenized[name] = docs
    vocabulary = {token for doc in docs for token in doc}
    rows.append({
        "pipeline": name,
        "vocabulary_size": len(vocabulary),
        "tokens_total": sum(len(doc) for doc in docs),
        "mean_tokens_per_doc": sum(len(doc) for doc in docs) / len(docs),
    })

pd.DataFrame(rows).set_index("pipeline").round(2)

In [ ]:
# --- Test 3.4: do not add, remove or modify anything in this cell ---
assert preprocess("The cats are RUNNING!") == ["the", "cats", "are", "running"]
assert preprocess("The cats are RUNNING!", remove_stops=True) == ["cats", "running"]
assert preprocess("The cats are RUNNING!", remove_stops=True, mode="stem") == ["cat", "run"], (
    f"got {preprocess('The cats are RUNNING!', remove_stops=True, mode='stem')}"
)
assert preprocess("The cats are RUNNING!", remove_stops=True, mode="lemma") == ["cat", "running"], (
    f"got {preprocess('The cats are RUNNING!', remove_stops=True, mode='lemma')}"
)
sizes = {name: len({t for doc in docs for t in doc}) for name, docs in train_tokenized.items()}
assert sizes["stemmed"] < sizes["no_stop"] < sizes["raw"], (
    f"Stemming should shrink the vocabulary the most: {sizes}"
)
print("Task 3.4 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">WRITTEN ANSWER &middot; BLOCK 2 OF 3 &middot; GRADED BY A HUMAN</span>

Scored between **0.0** and **1.0** (see *Grading* at the top). Quote *your* numbers and examples from this notebook; statements that would be true of any dataset score 0.0.

## Task 3.5 — Which pipeline would you choose?

Two questions. Both need some looking up; add scratch cells below the answer cell
for that.

### Question 1 — Where the standard advice is wrong for this task

"Remove stop words" is the textbook default, and Lecture 2 gives it too. But this task is about
detecting attacks on people. Look through `sorted(STOPWORDS)` and find one word that carries
real signal here. Quote it and name the linguistic pattern it belongs to. Write two short meme
texts of your own that differ only in that word and that you would label differently. Then
decide: keep it or drop it?

Two families of stop words are worth a look: negation and the second person. Check that the
word you pick is actually frequent in this data.

### Question 2 — A stemming collision, found by you

The Porter stemmer sometimes merges words that do not mean the same thing. Find such a
collision in your own vocabulary: two words with different meanings and the same stem. Show
both words and the stem. Then: does this collision cost you anything on this task? Say why, in
terms of what the classifier can and cannot distinguish now.

</div>


<div style="background:#EEF5FB;border:1px solid #4C72B0;border-left:9px solid #4C72B0;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#2A5183;letter-spacing:.05em;font-size:.9em">YOUR ANSWERS</span> &mdash; replace every *(write here)* line and keep the headings.

### Task 3.5 — question 1  (the stop word that carries signal)

**Word:** _(write here)_ — **pattern:** _(write here)_

- Text A: _(write here)_ → label I would give: _(write here)_
- Text B: _(write here)_ → label I would give: _(write here)_

**Keep it or drop it:** _(write here, with your reason)_

### Task 3.5 — question 2  (a stemming collision)

**Words:** _(write here)_ and _(write here)_ → **same stem:** _(write here)_

_(write here: does this collision cost you anything on this task, and why)_

</div>


---

# Part 4 — Dictionary and Bag of Words (15 points)

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 4.1 — Build the dictionary (4 points)

The dictionary $\mathbb{V} = \{v_1, \dots, v_N\}$ fixes which words become the columns of the
document-word matrix. Two rules from Lecture 2: the dictionary is built from training data
only (anything derived from dev or test is leakage), and words not in the dictionary are
**out-of-vocabulary (OOV)** and are mapped to `<oov>`.

Write `build_dictionary(tokenized_docs, min_count=2, max_size=None)`:
1. count how often every token occurs across all documents;
2. drop tokens occurring fewer than `min_count` times;
3. if `max_size` is given, keep only the `max_size` most frequent remaining tokens;
4. sort the kept tokens by count descending, then alphabetically, so the result is deterministic;
5. put `<oov>` at position 0, followed by the sorted tokens;
6. return a dictionary `token -> index` with indices `0, 1, 2, …`

</div>


In [ ]:
OOV_TOKEN = "<oov>"


def build_dictionary(tokenized_docs, min_count=2, max_size=None):
    """Build a token -> index mapping from tokenized training documents."""
    ### YOUR CODE HERE
    raise NotImplementedError()


vocab = build_dictionary(train_tokenized["stemmed"], min_count=2)
print(f"Dictionary size: {len(vocab)}")
print(f"First 10 entries: {list(vocab.items())[:10]}")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK 4.1, CONTINUED &middot; GRADED AUTOMATICALLY</span>

### How many words fall out of the dictionary?

The OOV rate tells you how much information you throw away on unseen data.

Write `oov_rate(tokenized_docs, vocabulary)` returning the fraction of all token occurrences that are not in the dictionary, a float between 0 and 1. This is a token-level rate: a rare word occurring once counts once. If there are no token occurrences at all (an empty corpus, or documents that are all empty), return `0.0` instead of dividing by zero.

</div>


In [ ]:
def oov_rate(tokenized_docs, vocabulary):
    """Fraction of token occurrences that are not in the dictionary."""
    ### YOUR CODE HERE
    raise NotImplementedError()


dev_tokenized = [preprocess(t, remove_stops=True, mode="stem") for t in df_dev["text"]]
test_tokenized = [preprocess(t, remove_stops=True, mode="stem") for t in df_test["text"]]

print(f"OOV rate on train: {oov_rate(train_tokenized['stemmed'], vocab):.2%}")
print(f"OOV rate on dev  : {oov_rate(dev_tokenized, vocab):.2%}")
print(f"OOV rate on test : {oov_rate(test_tokenized, vocab):.2%}")

for min_count in [1, 2, 5, 10]:
    v = build_dictionary(train_tokenized["stemmed"], min_count=min_count)
    print(f"min_count={min_count:>2}  dictionary={len(v):>6}  OOV on dev={oov_rate(dev_tokenized, v):6.2%}")

In [ ]:
# --- Test 4.1: do not add, remove or modify anything in this cell ---
toy_docs = [["a", "b", "c"], ["a", "b"], ["a"], ["d"]]   # counts: a=3, b=2, c=1, d=1
v = build_dictionary(toy_docs, min_count=2)
assert v == {"<oov>": 0, "a": 1, "b": 2}, f"got {v}"
assert list(v)[0] == "<oov>" and v["<oov>"] == 0, "<oov> must be the first entry, with index 0."

v_all = build_dictionary(toy_docs, min_count=1)
assert list(v_all) == ["<oov>", "a", "b", "c", "d"], (
    f"Sort by count descending, then alphabetically. Got {list(v_all)}"
)
v_small = build_dictionary(toy_docs, min_count=1, max_size=2)
assert list(v_small) == ["<oov>", "a", "b"], f"max_size must keep the most frequent. Got {list(v_small)}"
assert sorted(vocab.values()) == list(range(len(vocab))), "Indices must be 0..N-1 without gaps."

toy_vocab = {"<oov>": 0, "a": 1, "b": 2}
assert oov_rate([["a", "b"]], toy_vocab) == 0.0
assert oov_rate([["a", "z"]], toy_vocab) == 0.5, "One of two tokens is OOV -> 0.5"
assert oov_rate([["z", "z", "z", "a"]], toy_vocab) == 0.75
assert oov_rate([], toy_vocab) == 0.0, "No tokens at all -> 0.0, not a crash."
assert oov_rate([[], []], toy_vocab) == 0.0, "Documents that are all empty -> also 0.0, not a division by zero."
assert 0.0 < oov_rate(dev_tokenized, vocab) < 0.3, "The OOV rate on dev should be a small positive fraction."
print("Task 4.1 passed")

Raising `min_count` shrinks the dictionary (fewer features, less overfitting, less memory)
and raises the OOV rate (more information thrown away). That is the "remove very rare words"
knob from Lecture 2. There is no universally right value; you pick it on the validation set.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 4.2 — Term weighting, by hand (3 points)

The weighting models from Lecture 2, first on a tiny corpus where you can check every number
by hand. For a corpus given as a list of token lists:

$$\mathrm{tc}_{v,d} = \text{number of occurrences of } v \text{ in } d$$

$$\mathrm{tf}_{v,d} = \log(1 + \mathrm{tc}_{v,d}) \qquad\text{(the log-dampened version)}$$

$$\mathrm{idf}_v = \log \frac{M}{\mathrm{df}_v} \qquad \mathrm{df}_v = \#\{d : v \in d\}$$

$$\mathrm{tfidf}_{v,d} = \mathrm{tf}_{v,d} \times \mathrm{idf}_v$$

Write three functions:
- `term_counts(doc)` → a `collections.Counter` of token → count for one document. A real `Counter`, not a plain `dict`: a token that does not occur must give `0`, not raise.
- `document_frequencies(docs)` → `Counter` of token → number of documents it occurs in (not how often in total).
- `idf(docs)` → dict of token → `log(M / df)`, natural logarithm, with `M` the number of documents.

</div>


In [ ]:
def term_counts(doc):
    """Counter of token -> number of occurrences, for a single tokenized document."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def document_frequencies(docs):
    """Counter of token -> number of documents containing it."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def idf(docs):
    """Dict of token -> log(M / df), with M the number of documents."""
    ### YOUR CODE HERE
    raise NotImplementedError()


# A tiny corpus you can verify with pen and paper.
toy_corpus = [
    ["jku", "linz", "linz"],
    ["university", "linz"],
    ["university", "of", "vienna"],
]
print("term counts of doc 0:", dict(term_counts(toy_corpus[0])))
print("document frequencies:", dict(document_frequencies(toy_corpus)))
print("idf:", {k: round(v, 3) for k, v in idf(toy_corpus).items()})

In [ ]:
# --- Test 4.2: do not add, remove or modify anything in this cell ---
tc = term_counts(toy_corpus[0])
assert tc["linz"] == 2 and tc["jku"] == 1, f"got {dict(tc)}"
assert isinstance(tc, Counter), f"term_counts must return a collections.Counter, got {type(tc).__name__}."
assert term_counts([])["anything"] == 0, "A missing token must give 0, not raise -- that is what Counter is for."

df_counts = document_frequencies(toy_corpus)
assert df_counts["linz"] == 2, f"'linz' occurs in 2 documents (3 times in total!) — got {df_counts['linz']}."
assert df_counts["university"] == 2, f"got {df_counts['university']}"
assert df_counts["jku"] == 1 and df_counts["vienna"] == 1

idf_values = idf(toy_corpus)
assert abs(idf_values["linz"] - np.log(3 / 2)) < 1e-9, "idf(linz) must be log(3/2)."
assert abs(idf_values["jku"] - np.log(3 / 1)) < 1e-9, "idf(jku) must be log(3/1)."
assert idf_values["jku"] > idf_values["linz"], "A rarer word must get a HIGHER idf."
print("Task 4.2 passed")

`jku` occurs in one of three documents and gets $\log 3 \approx 1.10$, `linz` occurs in two and
gets $\log 1.5 \approx 0.41$: the rarer word weighs more. tf-idf therefore down-weights
frequent, uninformative words instead of deleting them with a stop word list.

### Background: what the formulas do (nothing to implement)

The next cell plots `tc`, `tf`, `idf`, and the BM25 term factor you implement in Task 4.4.


In [ ]:
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(15, 4.2))

# --- 1. term frequency: raw counts versus the log-damped version ---
counts = np.arange(0, 21)
left.plot(counts, counts, "o-", label="raw count  $tc_{v,d}$", color="#C44E52")
left.plot(counts, np.log(1 + counts), "o-", label=r"$\log(1 + tc_{v,d})$", color="#4C72B0")
left.set_xlabel("occurrences of the word in the document")
left.set_ylabel("weight")
left.set_title("Damping: raw count against log(1 + tc)")
left.legend()
left.grid(alpha=0.3)

# --- 2. idf: how the weight falls as a word spreads across the collection ---
M = len(df_train)
document_frequency = np.arange(1, M)
middle.plot(document_frequency, np.log(M / document_frequency), color="#55A868", linewidth=2)
train_df_counts = document_frequencies([t.lower().split() for t in df_train["text"]])
for word in ["the", "you", "when", "black", "muslim", "coffee"]:
    df_value = train_df_counts.get(word, 1)
    middle.plot(df_value, np.log(M / df_value), "o", color="#C44E52", markersize=7)
    middle.annotate(f" {word}", (df_value, np.log(M / df_value)), fontsize=9, color="#C44E52")
middle.set_xscale("log")
middle.set_xlabel("document frequency $df_v$  (log)")
middle.set_ylabel(r"$idf_v = \log(M / df_v)$")
middle.set_title("idf: rare words are worth more")
middle.grid(alpha=0.3, which="both")

# --- 3. BM25 saturation, the model you implement next ---
for k1 in [0.5, 1.2, 2.0, 8.0]:
    right.plot(counts, (k1 + 1) * counts / (k1 + counts), label=f"BM25, $k_1$={k1}")
right.plot(counts, np.log(1 + counts), "--", color="grey", label=r"$\log(1+tc)$")
right.set_xlabel("occurrences of the word in the document")
right.set_ylabel("term factor")
right.set_title(r"BM25 saturation: $k_1$ sets the ceiling")
right.legend(fontsize=8)
right.grid(alpha=0.3)

plt.tight_layout()
plt.show()

- **Left:** with $\log(1 + tc)$ the 20th occurrence of a word adds almost nothing over the 19th.
- **Middle:** idf with real words from your training set. *the* gets a low weight,
  *coffee* a high one; idf down-weights the stop words without being told.
- **Right:** BM25 saturates: the factor never exceeds $k_1 + 1$. Small $k_1$ means "once is
  enough", large $k_1$ is almost linear.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 4.3 — The document-word matrix (4 points)

Now build the real matrix $X$ of shape $(M \times N)$: one row per document, one column per
dictionary entry. Most entries are zero, so store it as a **sparse matrix** (`scipy.sparse`
keeps only the non-zero values). With 8500 documents and about 3800 dictionary entries, a
dense matrix would spend over 99% of its memory on zeros.

Write `build_matrix(tokenized_docs, vocabulary, idf_values=None)`:
- if `idf_values is None`, entry `(d, v)` is `tf = log(1 + tc)`;
- otherwise it is `tf * idf` (use `0.0` for tokens missing from `idf_values`);
- every OOV token maps to column 0, the `<oov>` column, and their counts add up there;
- return a `scipy.sparse.csr_matrix` of shape `(len(tokenized_docs), len(vocabulary))`.

</div>


In [ ]:
from scipy.sparse import csr_matrix


def build_matrix(tokenized_docs, vocabulary, idf_values=None):
    """Build the sparse document-word matrix (tf, or tf-idf if idf_values is given)."""
    ### YOUR CODE HERE
    raise NotImplementedError()


train_idf = idf(train_tokenized["stemmed"])

X_train = build_matrix(train_tokenized["stemmed"], vocab, train_idf)
X_dev = build_matrix(dev_tokenized, vocab, train_idf)
X_test = build_matrix(test_tokenized, vocab, train_idf)

y_train = df_train["label"].to_numpy()
y_dev = df_dev["label"].to_numpy()
y_test = df_test["label"].to_numpy()

density = X_train.nnz / (X_train.shape[0] * X_train.shape[1])
print(f"X_train: {X_train.shape[0]} documents x {X_train.shape[1]} features")
print(f"non-zero entries: {X_train.nnz:,}  ->  {density:.3%} of the matrix is filled")
print(f"as a dense matrix this would need {X_train.shape[0] * X_train.shape[1] * 8 / 1e6:.0f} MB")

In [ ]:
# --- Test 4.3: do not add, remove or modify anything in this cell ---
from scipy.sparse import issparse

toy_vocab = {"<oov>": 0, "a": 1, "b": 2}
M = build_matrix([["a", "a", "b"], ["zzz", "zzz"]], toy_vocab)
assert issparse(M), "build_matrix must return a scipy sparse matrix."
assert isinstance(M, csr_matrix), (
    f"build_matrix must return a csr_matrix (not {type(M).__name__}); csr_array indexes "
    "differently and breaks Part 5."
)
assert M.shape == (2, 3), f"Shape should be (2, 3), got {M.shape}."
D = M.toarray()
assert abs(D[0, 1] - np.log(3)) < 1e-9, f"tf of 'a' (2 occurrences) is log(1+2), got {D[0, 1]}."
assert abs(D[0, 2] - np.log(2)) < 1e-9, f"tf of 'b' (1 occurrence) is log(1+1), got {D[0, 2]}."
assert abs(D[1, 0] - np.log(3)) < 1e-9, (
    f"Both OOV tokens must be summed into column 0 -> log(1+2)={np.log(3):.4f}, got {D[1, 0]}. "
    "Did you add one entry per token instead of counting first?"
)

Mi = build_matrix([["a", "a"]], toy_vocab, {"a": 2.0}).toarray()
assert abs(Mi[0, 1] - np.log(3) * 2.0) < 1e-9, "With idf the value must be tf * idf."

assert X_train.shape == (8500, len(vocab))
assert X_dev.shape == (500, len(vocab)) and X_test.shape == (1000, len(vocab))
assert X_train.nnz < X_train.shape[0] * X_train.shape[1] * 0.01, "The matrix should be very sparse."
print("Task 4.3 passed")

### What "sparse" looks like

The next cell draws the first 300 documents against the 300 most frequent dictionary words,
the densest corner of the whole matrix. Every dot is a non-zero entry.


In [ ]:
block = X_train[:300, :300].toarray()

fig, ax = plt.subplots(figsize=(7, 7))
ax.spy(block, markersize=1.4, color="#4C72B0", aspect="auto")
ax.set_xlabel("dictionary index (the 300 most frequent words)")
ax.set_ylabel("document")
ax.set_title(f"Non-zero entries: {(block != 0).sum()} of {block.size} "
             f"({(block != 0).mean():.1%} filled)")
plt.tight_layout()
plt.show()


Almost everything is white. Each document touches about six of several thousand
dimensions, so two documents are nearly orthogonal unless they share exact words, and related
words earn no partial credit. Word embeddings address this in Assignment 2.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 4.4 — BM25 (4 points)

BM25 refines tf-idf with the two ideas from the right-hand plot above:

$$x_{v,d} = \underbrace{\frac{(k_1 + 1)\, tc_{v,d}}{k_1\left(1 - b + b\frac{|d|}{avgdl}\right) + tc_{v,d}}}_{\text{saturating, length-normalized term factor}} \times\; idf_v$$

- $|d|$ is the length of document $d$ in tokens, $avgdl$ the average document length across
  the documents passed in (`tokenized_docs`);
- $k_1$ sets how fast the term factor saturates (typical: 1.2 to 2.0);
- $b$ sets how strongly document length is normalized: $b=0$ ignores length, $b=1$
  normalizes fully.

Write two functions:
- `bm25_term_factor(tc, doc_len, avgdl, k1=1.5, b=0.75)`: only the underbraced factor, a single float. Return `0.0` when `tc` is 0.
- `build_bm25_matrix(tokenized_docs, vocabulary, idf_values, k1=1.5, b=0.75)`: the same shape and OOV rule as `build_matrix`, but each entry is `bm25_term_factor(...) * idf_values[token]` (use `0.0` for tokens missing from `idf_values`).

</div>


In [ ]:
def bm25_term_factor(tc, doc_len, avgdl, k1=1.5, b=0.75):
    """The saturating, length-normalized term factor of BM25."""
    ### YOUR CODE HERE
    raise NotImplementedError()


def build_bm25_matrix(tokenized_docs, vocabulary, idf_values, k1=1.5, b=0.75):
    """Document-word matrix weighted with BM25."""
    ### YOUR CODE HERE
    raise NotImplementedError()


X_train_bm25 = build_bm25_matrix(train_tokenized["stemmed"], vocab, train_idf)
print(f"BM25 matrix: {X_train_bm25.shape}, {X_train_bm25.nnz:,} non-zeros")

In [ ]:
# --- Test 4.4: do not add, remove or modify anything in this cell ---
# With b=0 the length term drops out: factor = (k1+1)*tc / (k1 + tc).
assert abs(bm25_term_factor(1, 10, 10, k1=1.5, b=0.0) - 2.5 / 2.5) < 1e-9
assert abs(bm25_term_factor(3, 10, 10, k1=1.5, b=0.0) - (2.5 * 3) / (1.5 + 3)) < 1e-9
assert bm25_term_factor(0, 10, 10) == 0.0, "a word that does not occur must give 0.0"

# A document of average length behaves like the b=0 case, whatever b is.
assert abs(bm25_term_factor(4, 12, 12, k1=1.2, b=0.75)
           - (2.2 * 4) / (1.2 + 4)) < 1e-9, "at doc_len == avgdl the b term must cancel"

# Saturation: the factor can never exceed k1 + 1.
assert bm25_term_factor(10_000, 10, 10, k1=1.5, b=0.0) < 2.5
assert bm25_term_factor(10, 10, 10) > bm25_term_factor(1, 10, 10), "more occurrences -> larger"

# A long document is penalized compared with a short one holding the same count.
assert bm25_term_factor(3, 50, 10, b=0.75) < bm25_term_factor(3, 5, 10, b=0.75)

toy_vocab = {"<oov>": 0, "p": 1, "q": 2}
M_bm25 = build_bm25_matrix([["p", "p"], ["q"]], toy_vocab, {"p": 2.0, "q": 1.0})
assert M_bm25.shape == (2, 3)
dense_bm25 = M_bm25.toarray()
expected = bm25_term_factor(2, 2, 1.5) * 2.0
assert abs(dense_bm25[0, 1] - expected) < 1e-9, (
    f"expected {expected:.4f} (avgdl of [2, 1] is 1.5), got {dense_bm25[0, 1]:.4f}"
)
assert X_train_bm25.shape == X_train.shape
print("Task 4.4 passed")

In [ ]:
# --- BM25 versus tf-idf on the longest training document (nothing to implement) ---
index_to_word = {index: word for word, index in vocab.items()}
stemmed_docs = train_tokenized["stemmed"]
avgdl = sum(len(d) for d in stemmed_docs) / len(stemmed_docs)

# The longest training document: long enough to show both saturation and the length penalty.
doc = max(range(len(stemmed_docs)), key=lambda i: len(stemmed_docs[i]))


def top_words(matrix, k=6):
    row = matrix[doc].toarray()[0]
    order = np.argsort(-row)
    return [(index_to_word[i], round(float(row[i]), 2)) for i in order[:k] if row[i] > 0]


print(f'document {doc}: "{df_train["text"].iloc[doc][:90]}..."')
print(f"length {len(stemmed_docs[doc])} tokens against an average of {avgdl:.1f} "
      f"-> BM25 length bracket = {1 - 0.75 + 0.75 * len(stemmed_docs[doc]) / avgdl:.2f}\n")
print("tf-idf :", top_words(X_train))
print("BM25   :", top_words(X_train_bm25))


The ranking of words inside the document barely changes. BM25 caps repeated words and
discounts this document for being longer than average. With the lengths from Part 2 (median 10
words, mean 11.7, range 1 to 70) the bracket $1 - b + b\,|d|/avgdl$ is close to 1 for a typical
meme, so BM25 differs little from tf-idf here. On news articles, where lengths vary far more,
the same parameter changes a lot.


---

# Part 5 — Classification and evaluation (12 points)

$X$ and $y$ are in place. The rest is the standard ML pipeline from the end of Lecture 2.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 5.1 — Train and evaluate a classifier (3 points)

The next cell trains a **logistic regression** on the training matrix and evaluates it on dev.

Write `evaluate(model, X, y)` returning a dictionary with `"accuracy"`, `"precision"`, `"recall"` and `"f1"`, computed with the hateful class (label 1) as the positive class. A model that predicts no positives at all must score `0` on precision, recall and F1, without raising or warning.

</div>


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def evaluate(model, X, y):
    """Return accuracy, precision, recall and F1 (positive class = 1) as a dict."""
    ### YOUR CODE HERE
    raise NotImplementedError()


model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train, y_train)

scores_train = evaluate(model, X_train, y_train)
scores_dev = evaluate(model, X_dev, y_dev)

print(f"{'':<12}{'accuracy':>10}{'precision':>11}{'recall':>9}{'f1':>8}")
for name, s in [("train", scores_train), ("dev", scores_dev)]:
    print(f"{name:<12}{s['accuracy']:>10.3f}{s['precision']:>11.3f}{s['recall']:>9.3f}{s['f1']:>8.3f}")
print(f"\nmajority baseline on dev: {majority_baseline_accuracy(df_train, df_dev):.3f}")

In [ ]:
# --- Test 5.1: do not add, remove or modify anything in this cell ---
assert set(scores_dev) == {"accuracy", "precision", "recall", "f1"}, f"Wrong keys: {sorted(scores_dev)}"


class _AlwaysOne:
    def predict(self, X):
        return np.ones(X.shape[0], dtype=int)


# Half the labels are 1, so always predicting 1 gives accuracy 0.5, recall 1.0, precision 0.5.
_y = np.array([0, 1, 0, 1])
_s = evaluate(_AlwaysOne(), np.zeros((4, 1)), _y)
assert abs(_s["accuracy"] - 0.5) < 1e-9, f"accuracy should be 0.5, got {_s['accuracy']}"
assert abs(_s["recall"] - 1.0) < 1e-9, f"recall should be 1.0, got {_s['recall']}"
assert abs(_s["precision"] - 0.5) < 1e-9, f"precision should be 0.5, got {_s['precision']}"
assert scores_train["accuracy"] > scores_dev["accuracy"], (
    "Training accuracy is normally higher than dev accuracy here."
)
print("Task 5.1 passed")

Training accuracy is far above dev accuracy: the model overfits (many features, few examples
per feature). And dev accuracy is only a little above the 50% baseline. That is not a bug in
your code; Part 6 comes back to it.

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 3 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 5.2 — Model selection on the validation set (3 points)

Try several configurations, pick the best on the validation set, and only then touch the test
set. Vary the preprocessing pipeline (`"no_stop"`, `"stemmed"`, `"lemmatized"`) and the
regularization strength `C` of the logistic regression (`0.1`, `1.0`, `10.0`; smaller `C`
means stronger regularization and less overfitting).

Write `run_configuration(pipeline_name, C, min_count=2)` that runs the whole pipeline for one configuration and returns a dict with `"pipeline"`, `"C"`, `"dev_accuracy"`, `"dev_f1"`, and `"model"`, `"vocab"`, `"idf"` (needed later). Dictionary and idf must be rebuilt from the training set only, for each configuration.

</div>


In [ ]:
def run_configuration(pipeline_name, C, min_count=2):
    """Train and evaluate one full configuration; return its dev scores and the fitted parts."""
    ### YOUR CODE HERE
    raise NotImplementedError()


results = []
for pipeline_name in ["no_stop", "stemmed", "lemmatized"]:
    for C in [0.1, 1.0, 10.0]:
        result = run_configuration(pipeline_name, C)
        results.append(result)
        print(f"{pipeline_name:<12} C={C:<6} dev accuracy={result['dev_accuracy']:.3f}  "
              f"dev F1={result['dev_f1']:.3f}")

results_df = pd.DataFrame(results)[["pipeline", "C", "dev_accuracy", "dev_f1"]]
best = max(results, key=lambda r: r["dev_accuracy"])
print(f"\nBest configuration: {best['pipeline']}, C={best['C']} "
      f"(dev accuracy {best['dev_accuracy']:.3f})")
results_df.sort_values("dev_accuracy", ascending=False).round(3)

Only now, with the configuration fixed, may you look at the test set (Lecture 2: select on
the validation set, report on the test set). The next cell rebuilds the winning
configuration's features for the test set and reports the final score.


In [ ]:
# Rebuild the winning configuration's features for the test set and report the final score.
best_kwargs = PIPELINES[best["pipeline"]]
test_docs = [preprocess(t, **best_kwargs) for t in df_test["text"]]
X_test_best = build_matrix(test_docs, best["vocab"], best["idf"])

test_scores = evaluate(best["model"], X_test_best, y_test)

print(f"FINAL test scores ({best['pipeline']}, C={best['C']}):")
for metric, value in test_scores.items():
    print(f"  {metric:<10}{value:.3f}")
print(f"\n  majority baseline on test: {majority_baseline_accuracy(df_train, df_test):.3f}")

In [ ]:
# --- Test 5.2: do not add, remove or modify anything in this cell ---
assert len(results) == 9, f"You should have tried 3 pipelines x 3 values of C = 9 configurations, got {len(results)}."
assert set(results[0]) >= {"pipeline", "C", "dev_accuracy", "dev_f1", "model", "vocab", "idf"}, (
    f"run_configuration is missing keys: {sorted(results[0])}"
)
assert all(0.0 <= r["dev_accuracy"] <= 1.0 for r in results)
assert best["dev_accuracy"] == max(r["dev_accuracy"] for r in results)
assert test_scores["accuracy"] > 0.5, "The final model should at least beat the 50% baseline on test."
print("Task 5.2 passed")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 4 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 5.3 — What did the model learn, and why did it say that? (4 points)

A linear model is interpretable: each dictionary word has one weight. A large positive weight
pushes towards "hateful", a large negative one towards "not hateful".

Write `top_weighted_words(model, vocabulary, n=15)` returning a DataFrame with the columns `word` and `weight`, containing the `n` most positive and the `n` most negative weights, sorted by weight descending.

</div>


In [ ]:
def top_weighted_words(model, vocabulary, n=15):
    """The n most positive and n most negative model weights, as a DataFrame."""
    ### YOUR CODE HERE
    raise NotImplementedError()


weights = top_weighted_words(best["model"], best["vocab"])
print("Pushes towards HATEFUL:")
display(weights.head(15))
print("Pushes towards NOT HATEFUL:")
display(weights.tail(15).iloc[::-1])

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK 5.3, CONTINUED &middot; GRADED AUTOMATICALLY</span>

### Why did the model say that?

The question about any deployed classifier is "why did you flag this particular post?". For a
linear model the answer is exact: the decision is a sum of per-word contributions.

$$z = b + \sum_{v \in d} \underbrace{w_v \cdot x_{v,d}}_{\text{contribution of word } v}$$

Every word contributes its weight times its feature value. Positive pushes towards *hateful*,
negative towards *not hateful*. The bias $b$ belongs to no word and is not part of the table
you build.

Write `explain_prediction(model, vocabulary, x_row)`, where `x_row` is one row of a document-word matrix (a sparse matrix of shape `(1, N)`). Return a DataFrame with the columns
- `word`: the dictionary word
- `value`: its feature value in this document
- `weight`: the model weight
- `contribution`: `value * weight`

containing only the words present in this document (non-zero feature value), sorted by `contribution` descending.

</div>


In [ ]:
def explain_prediction(model, vocabulary, x_row):
    """Per-word contributions to one prediction, as a DataFrame sorted by contribution."""
    ### YOUR CODE HERE
    raise NotImplementedError()


# Explain the dev meme the model is most confident is hateful.
# The dev features are rebuilt with the winning pipeline, as for the test set above.
dev_docs_best = [preprocess(t, **best_kwargs) for t in df_dev["text"]]
X_dev_best = build_matrix(dev_docs_best, best["vocab"], best["idf"])
probabilities = best["model"].predict_proba(X_dev_best)[:, 1]
most_confident = int(np.argmax(probabilities))
explanation = explain_prediction(best["model"], best["vocab"],
                                 X_dev_best[most_confident:most_confident + 1])

print(f'text : "{df_dev["text"].iloc[most_confident]}"')
print(f"true label: {y_dev[most_confident]}   predicted P(hateful) = {probabilities[most_confident]:.2f}\n")
display(explanation)

In [ ]:
# --- Test 5.3: do not add, remove or modify anything in this cell ---
assert list(weights.columns) == ["word", "weight"], f"Columns must be ['word', 'weight'], got {list(weights.columns)}."
assert len(weights) == 30, f"15 most positive + 15 most negative = 30 rows, got {len(weights)}."
assert weights["weight"].is_monotonic_decreasing, "Sort by weight, descending."
assert weights["weight"].iloc[0] > 0 > weights["weight"].iloc[-1]
assert weights["word"].nunique() == 30, "The words should all be different."

class _FakeModel:
    coef_ = np.array([[0.0, 2.0, -3.0, 1.0]])

_vocab = {"<oov>": 0, "up": 1, "down": 2, "absent": 3}
_row = csr_matrix(np.array([[0.0, 1.5, 2.0, 0.0]]))
_explained = explain_prediction(_FakeModel(), _vocab, _row)

assert list(_explained.columns) == ["word", "value", "weight", "contribution"], (
    f"columns must be ['word', 'value', 'weight', 'contribution'], got {list(_explained.columns)}"
)
assert len(_explained) == 2, f"only words present in the document -> 2 rows, got {len(_explained)}"
assert set(_explained["word"]) == {"up", "down"}, "words with value 0 must be left out"
assert _explained["contribution"].is_monotonic_decreasing, "sort by contribution, descending"
assert abs(_explained.iloc[0]["contribution"] - 3.0) < 1e-9, "1.5 * 2.0 = 3.0 should come first"
assert abs(_explained.iloc[-1]["contribution"] + 6.0) < 1e-9, "2.0 * -3.0 = -6.0 should come last"
print("Task 5.3 passed")

The next cell draws the contributions as a bar chart, once for the dev meme the model is most
confident about and once for a meme it got wrong. This is the kind of picture a moderation
team would be shown before trusting a model.


In [ ]:
def plot_explanation(explanation, text, probability, top_n=12):
    """Horizontal bar chart of the words that drove one decision."""
    strongest = explanation.reindex(
        explanation["contribution"].abs().sort_values(ascending=False).index
    ).head(top_n).sort_values("contribution")

    colors = ["#C44E52" if c > 0 else "#4C72B0" for c in strongest["contribution"]]
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(strongest) + 1.6))
    ax.barh(strongest["word"], strongest["contribution"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("contribution to the score   (weight x feature value)")
    ax.set_title(f'P(hateful) = {probability:.2f}\n"{text[:70]}"', fontsize=10)
    fig.text(0.99, 0.02, "red = towards hateful    blue = towards not hateful",
             ha="right", fontsize=8, color="grey")
    plt.tight_layout()
    plt.show()


plot_explanation(explanation, df_dev["text"].iloc[most_confident], probabilities[most_confident])

# The same plot for the dev meme the model got most wrong.
predictions = best["model"].predict(X_dev_best)
mistakes = np.where(predictions != y_dev)[0]
worst = mistakes[np.argmax(np.abs(probabilities[mistakes] - y_dev[mistakes]))]
plot_explanation(
    explain_prediction(best["model"], best["vocab"], X_dev_best[worst:worst + 1]),
    df_dev["text"].iloc[worst],
    probabilities[worst],
)
print(f"True label {y_dev[worst]}, predicted P(hateful) = {probabilities[worst]:.2f}.")

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">TASK &middot; 2 POINTS &middot; GRADED AUTOMATICALLY</span>

## Task 5.4 — Choosing an operating point (2 points)

`model.predict()` uses a threshold of 0.5: flag the meme if $P(\text{hateful}) \geq 0.5$. That
threshold is a policy decision, and the most consequential setting in a moderation system. A
low threshold flags more content: higher recall, more wrongly deleted posts. A high threshold
flags less: higher precision, more hateful content left up.

Write `precision_recall_table(model, X, y, thresholds)` returning a DataFrame with one row per threshold, in the order given, and the columns `threshold`, `precision`, `recall`, `f1`, `n_flagged` (how many items were predicted hateful at that threshold). A threshold that flags nothing must give precision, recall and F1 of `0` rather than an error.

</div>


In [ ]:
def precision_recall_table(model, X, y, thresholds):
    """Precision, recall, F1 and the number of flagged items at each decision threshold."""
    ### YOUR CODE HERE
    raise NotImplementedError()


thresholds = np.round(np.arange(0.05, 1.0, 0.05), 2)
pr_table = precision_recall_table(best["model"], X_dev_best, y_dev, thresholds)
pr_table.round(3)

In [ ]:
# --- Test 5.4: do not add, remove or modify anything in this cell ---
class _ProbaModel:
    def __init__(self, p):
        self.p = np.asarray(p)

    def predict_proba(self, X):
        return np.column_stack([1 - self.p, self.p])


_y = np.array([1, 1, 0, 0])
_m = _ProbaModel([0.9, 0.6, 0.4, 0.1])
_t = precision_recall_table(_m, np.zeros((4, 1)), _y, [0.5, 0.95])

assert list(_t.columns) == ["threshold", "precision", "recall", "f1", "n_flagged"], (
    f"wrong columns: {list(_t.columns)}"
)
assert len(_t) == 2
# At 0.5 the two positives are flagged and nothing else: perfect precision and recall.
assert abs(_t.loc[0, "precision"] - 1.0) < 1e-9 and abs(_t.loc[0, "recall"] - 1.0) < 1e-9
assert _t.loc[0, "n_flagged"] == 2
# At 0.95 nothing is flagged: precision and recall are 0 (and it must not crash).
assert _t.loc[1, "n_flagged"] == 0
assert _t.loc[1, "recall"] == 0.0 and _t.loc[1, "precision"] == 0.0
print("Task 5.4 passed")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(pr_table["threshold"], pr_table["precision"], "o-", label="precision", color="#C44E52")
left.plot(pr_table["threshold"], pr_table["recall"], "o-", label="recall", color="#4C72B0")
left.plot(pr_table["threshold"], pr_table["f1"], "o-", label="F1", color="#55A868")
left.axvline(0.5, color="grey", linestyle="--", linewidth=1)
left.annotate("the default 0.5", (0.5, 0.05), fontsize=9, color="grey", rotation=90)
left.set_xlabel("decision threshold")
left.set_ylabel("score")
left.set_title("Precision, recall and F1 by threshold")
left.legend()
left.grid(alpha=0.3)

right.plot(pr_table["recall"], pr_table["precision"], "o-", color="#4C72B0")
for _, row in pr_table.iloc[::3].iterrows():
    right.annotate(f' {row["threshold"]:.2f}', (row["recall"], row["precision"]), fontsize=8)
right.axhline(y_dev.mean(), color="grey", linestyle="--", linewidth=1)
right.annotate("always-flag baseline", (0.35, y_dev.mean() + 0.01), fontsize=8, color="grey")
right.set_xlabel("recall  (share of hateful memes caught)")
right.set_ylabel("precision  (share of flagged memes that really are hateful)")
right.set_title("Precision-recall trade-off")
right.grid(alpha=0.3)

plt.tight_layout()
plt.show()

usable = pr_table[pr_table["precision"] >= 0.8]
if len(usable):
    row = usable.loc[usable["recall"].idxmax()]
    print(f"If the policy demands precision >= 0.80, the best threshold is "
          f"{row['threshold']:.2f} and it catches {row['recall']:.1%} of hateful memes.")
else:
    print("This model NEVER reaches precision 0.80 at any threshold — "
          "at every operating point, at least one in five flagged memes is innocent.")

---

# Part 6 — Reflection

<div style="background:#FFF4E5;border:1px solid #E8A33D;border-left:9px solid #E8A33D;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#9C5A00;letter-spacing:.05em;font-size:.9em">WRITTEN ANSWER &middot; BLOCK 3 OF 3 &middot; GRADED BY A HUMAN</span>

Scored between **0.0** and **1.0** (see *Grading* at the top). Quote *your* numbers and examples from this notebook; statements that would be true of any dataset score 0.0.

Three questions. This block is not about code.

### Question 1 — Your prediction, and where your pipeline loses information

Quote three numbers: your prediction from Task 2.4 question 2, the majority baseline, and your
actual test accuracy from Task 5.2. Were you too optimistic or too pessimistic? Then give two
distinct reasons why your accuracy is not higher. The reasons must be about bag-of-words and
your pipeline, not about the missing image, and for each reason name where in the pipeline the
information is lost: which function, which step of
`raw text → normalize → tokenize → dictionary → matrix → classifier`. "The model is too
simple" is not a reason.

### Question 2 — Your operating point, in people

From your Task 5.4 table, pick one row and quote its `threshold`, `precision` and `recall`
exactly as printed. Convert that row into people: of 1,000 posts this system deletes, how many
belonged to someone who did nothing wrong? Of 100 hateful posts, how many stay up? Then:
would you ship this? If not, state the precision you would demand and read off your own table
what recall that leaves you.

### Question 3 — The harm your model would do

From your top-weighted words in Task 5.3, pick the one that worries you most. Write one
ordinary, harmless sentence that a person from the targeted group might post and that your
model would flag as hateful. Then say who that person is and what they lose when the post
disappears. One paragraph, no abstractions; the words "bias" and "fairness" on their own earn
nothing.

</div>


<div style="background:#EEF5FB;border:1px solid #4C72B0;border-left:9px solid #4C72B0;border-radius:6px;padding:6px 20px 14px 20px;margin:14px 0 14px 0;color:#1a1a1a">

<span style="font-weight:700;color:#2A5183;letter-spacing:.05em;font-size:.9em">YOUR ANSWERS</span> &mdash; replace every *(write here)* line and keep the headings.

### Part 6 — question 1  (your prediction, and where your pipeline loses information)

**Predicted in Task 2.4:** _(write here)_ · **Majority baseline:** _(write here)_ · **My test accuracy:** _(write here)_

_(write here: too optimistic or too pessimistic)_

**Reason 1** — _(write here)_

Where the information is lost: _(write here)_

**Reason 2** — _(write here)_

Where the information is lost: _(write here)_

### Part 6 — question 2  (your operating point, in people)

**Chosen row:** threshold = _(write here)_, precision = _(write here)_, recall = _(write here)_

- Of 1,000 deleted posts, _(write here)_ belonged to someone who did nothing wrong.
- Of 100 hateful posts, _(write here)_ stay up.

**Would you ship it:** _(write here)_

### Part 6 — question 3  (the harm your own model would do)

**Word:** _(write here)_

**A harmless sentence my model would flag:** _(write here)_

_(write here: who that person is and what they lose)_

</div>


---

## Submission check

Run the next cell last. It is the same check the grader runs first: it verifies that the
notebook is complete. It does not tell you your grade.


In [ ]:
# --- Submission check: do not add, remove or modify anything in this cell ---
import inspect

REQUIRED_FUNCTIONS = [
    "load_dataframe",
    "label_distribution", "majority_baseline_accuracy",
    "class_association", "conflicting_texts",
    "normalize_text", "tokenize", "remove_stopwords",
    "stem_tokens", "lemmatize_tokens", "preprocess",
    "build_dictionary", "oov_rate",
    "term_counts", "document_frequencies", "idf", "build_matrix",
    "bm25_term_factor", "build_bm25_matrix",
    "evaluate", "run_configuration", "top_weighted_words",
    "explain_prediction", "precision_recall_table",
]

problems = []

for function_name in REQUIRED_FUNCTIONS:
    function = globals().get(function_name)
    if function is None:
        problems.append(f"the function {function_name}() is missing")
    elif not callable(function):
        problems.append(f"{function_name} exists but is not a function")

for variable in ["df_train", "df_dev", "df_test", "X_train", "vocab", "best", "test_scores"]:
    if variable not in globals():
        problems.append(f"the variable {variable} is missing — did every cell run?")

if problems:
    print("SUBMISSION NOT READY:\n")
    for problem in problems:
        print(f"  - {problem}")
    raise SystemExit("Fix the points above, then run all cells again.")

print("Submission check passed.\n")
print("  Save this notebook as:  assignment_01_<matriculation 1>_<matriculation 2>.ipynb")
print("  for example:            assignment_01_01234567_12345678.ipynb   (leading zeros included)")
print("  One file per team, uploaded by one team member.")

# This cell sees your code but not your prose, so it cannot check the written blocks.
print("\n  Not checked here: the written blocks (Tasks 2.4, 3.5 and Part 6).")
print("  Make sure every '_(write here)_' is gone and every number you quote is one")
print("  your notebook printed.")

---

## Before you submit

1. *Kernel → Restart Kernel and Run All Cells.* The notebook must run top to bottom without
   errors.
2. Every test cell prints `Task N.N passed`, and the submission check prints
   `Submission check passed.`
3. All three written blocks (Tasks 2.4, 3.5 and Part 6) are answered. Every `_(write here)_`
   is gone, and every number you quote is one your notebook printed.
4. Save the notebook with outputs. One team member uploads it to Moodle as
   `assignment_01_<matriculation 1>_<matriculation 2>.ipynb`.

## What comes next

| Problem you hit here | Addressed in |
|---|---|
| Words are independent dimensions; no notion of similar meaning | **Assignment 2**: word embeddings |
| Word order and context are thrown away | **Assignment 2**: n-gram language models; **Assignment 3**: attention and Transformers |
| Rare and unseen words carry no information | **Assignment 2**: fastText subword vectors |
| The picture is ignored, and Task 2.3 showed how many texts need it | **Assignments 3 and 4**: the image joins the text |

Keep this notebook. Assignment 2 starts from the bag-of-words baseline you built here.
